# Drought News Detector — Full LLM Pipeline

**Pipeline:** Scrape web news → ClimateBERT filter → LLM drought assessment → export CSV

Run cells top to bottom. All steps have fallbacks so pipeline never breaks.

## Step 1: Install Dependencies

In [ ]:
!pip install transformers torch feedparser newspaper3k huggingface_hub pandas tqdm requests

## Step 2: Imports & Config

In [ ]:
import os, json, time, urllib.request
import feedparser
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import torch
from transformers import pipeline as hf_pipeline

HF_TOKEN     = os.getenv('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')
LLM_MODEL    = 'mistralai/Mistral-7B-Instruct-v0.2'
CLIMATE_MODEL = 'climatebert/distilroberta-base-climate-detector'
print('Config set. HF token set:', HF_TOKEN != 'YOUR_HF_TOKEN_HERE')

## Step 3: Scrape Google News RSS

Uses browser user-agent to avoid blocks. Auto-loads sample data if network blocked.

In [ ]:
import ssl, urllib.request, feedparser, pandas as pd

HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'}

# Fix SSL certificate error on macOS
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

def google_news_rss(query, max_articles=20):
    query_enc = query.replace(' ', '+')
    url = f'https://news.google.com/rss/search?q={query_enc}&hl=en-US&gl=US&ceid=US:en'
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=15, context=ssl_ctx) as resp:
            raw = resp.read()
        feed = feedparser.parse(raw)
    except Exception as e:
        print(f'  Error: {e}')
        return []
    articles = []
    for entry in feed.entries[:max_articles]:
        src = entry.get('source', {})
        articles.append({
            'title':     entry.get('title', ''),
            'url':       entry.get('link', ''),
            'published': entry.get('published', ''),
            'source':    src.get('title', 'Unknown') if isinstance(src, dict) else str(src),
            'summary':   entry.get('summary', ''),
        })
    return articles

SAMPLE_ARTICLES = [
    {'title': 'Drought 2025 intensifies across Horn of Africa', 'url': 'sample-1', 'published': '2025-05-01', 'source': 'Reuters', 'summary': 'Severe water shortage leaves millions at risk as rainfall deficit continues across the region.'},
    {'title': 'California declares drought emergency as reservoirs drop to record low', 'url': 'sample-2', 'published': '2025-04-28', 'source': 'AP News', 'summary': 'State officials impose mandatory water restrictions as drought enters third consecutive year.'},
    {'title': 'Australia faces worst agricultural drought in decades', 'url': 'sample-3', 'published': '2025-04-25', 'source': 'BBC', 'summary': 'Farmers face crisis as soil moisture deficit leaves crops withered across Murray-Darling basin.'},
    {'title': 'Kenya declares national disaster as drought kills livestock', 'url': 'sample-4', 'published': '2025-04-20', 'source': 'Guardian', 'summary': 'Government seeks international aid as water scarcity forces communities to walk miles for water.'},
    {'title': 'Spain groundwater depletion accelerates amid prolonged dry spell', 'url': 'sample-5', 'published': '2025-04-18', 'source': 'Reuters', 'summary': 'Aquifer levels hit historic lows as meteorological drought enters second year across Iberian peninsula.'},
    {'title': 'Stock market hits record high on strong tech earnings', 'url': 'sample-6', 'published': '2025-05-01', 'source': 'Bloomberg', 'summary': 'Major indices rally as quarterly results beat analyst expectations across technology sector.'},
    {'title': 'New AI model released with improved reasoning capabilities', 'url': 'sample-7', 'published': '2025-04-30', 'source': 'TechCrunch', 'summary': 'Latest large language model shows improvements in mathematical reasoning benchmarks.'},
    {'title': 'Football championship final draws record television audience', 'url': 'sample-8', 'published': '2025-04-27', 'source': 'BBC Sport', 'summary': 'Over 50 million viewers tune in as home team clinches title in dramatic penalty shootout.'},
]

QUERIES = ['drought 2025', 'water shortage 2025', 'water scarcity crisis']
print('Scraping Google News RSS...')
all_articles = []
for q in QUERIES:
    results = google_news_rss(q, max_articles=15)
    all_articles.extend(results)
    print(f'  "{q}": {len(results)} articles')

if not all_articles:
    print('All feeds blocked — loading sample articles.')
    all_articles = SAMPLE_ARTICLES

df_raw = pd.DataFrame(all_articles).drop_duplicates(subset='url').reset_index(drop=True)
print(f'Total unique articles: {len(df_raw)}')
df_raw[['title', 'source', 'published']].head(10)


## Step 4: ClimateBERT Fast Filter

In [ ]:
print('Loading ClimateBERT...')
climate_pipe = hf_pipeline('text-classification', model=CLIMATE_MODEL, device=-1)
print('Loaded.')

def is_climate_related(text, threshold=0.6):
    try:
        r = climate_pipe(text[:512], truncation=True)[0]
        return r['label'] == 'yes' and r['score'] >= threshold
    except Exception:
        return True

df_raw['text_for_filter'] = df_raw['title'].fillna('') + ' ' + df_raw['summary'].fillna('')
df_raw['is_climate'] = [is_climate_related(t) for t in tqdm(df_raw['text_for_filter'])]
df_climate = df_raw[df_raw['is_climate']].reset_index(drop=True)
print(f'Passed filter: {len(df_climate)} / {len(df_raw)}')
df_climate[['title', 'source']].head(10)

## Step 5: Fetch Full Article Text

In [ ]:
def fetch_article_text(url):
    if not url:
        return ''
    try:
        from newspaper import Article
        art = Article(url)
        art.download()
        art.parse()
        return art.text[:2000]
    except Exception:
        return ''

print('Fetching article text...')
full_texts = []
for _, row in tqdm(df_climate.iterrows(), total=len(df_climate)):
    text = fetch_article_text(row['url'])
    if len(text) < 50:
        text = row.get('summary', '')
    full_texts.append(text)
    if row['url']:
        time.sleep(0.3)

df_climate = df_climate.copy()
df_climate['full_text'] = full_texts
print(f'Got text: {sum(len(t)>50 for t in full_texts)} / {len(df_climate)} articles')

## Step 6: LLM Drought Assessment

Requires HuggingFace token. Get free token at huggingface.co/settings/tokens → set HF_TOKEN in Step 2.

In [ ]:
from huggingface_hub import InferenceClient

if not HF_TOKEN or HF_TOKEN == 'YOUR_HF_TOKEN_HERE':
    print('No HF token. Get free token at https://huggingface.co/settings/tokens')
    print('Set HF_TOKEN in Step 2 then rerun Steps 6-7.')
else:
    client = InferenceClient(model=LLM_MODEL, token=HF_TOKEN)
    print(f'Client ready: {LLM_MODEL}')

FEW_SHOT = '''You are a drought news classifier. Respond with valid JSON only.

Example 1:
Article: "Severe drought threatens crops across southern Australia. Water reserves hit record lows."
Response: {"is_drought": true, "confidence": "high", "location": "southern Australia", "severity": "severe", "affected": "farmers", "key_quote": "water reserves hit record lows"}

Example 2:
Article: "Stock markets rally as earnings season beats expectations."
Response: {"is_drought": false, "confidence": "high", "location": null, "severity": null, "affected": null, "key_quote": null}
'''

def assess_drought(title, text):
    snippet = f'Title: {title}\nText: {text[:600]}'
    prompt = f'<s>[INST] {FEW_SHOT}\nClassify this article:\n{snippet} [/INST]'
    try:
        resp = client.text_generation(prompt, max_new_tokens=200, temperature=0.1)
        s, e = resp.find('{'), resp.rfind('}') + 1
        if s >= 0 and e > s:
            return json.loads(resp[s:e])
    except Exception as ex:
        return {'is_drought': None, 'error': str(ex)}
    return {'is_drought': None, 'error': 'no JSON'}

# Test on first article
if HF_TOKEN and HF_TOKEN != 'YOUR_HF_TOKEN_HERE' and len(df_climate) > 0:
    row = df_climate.iloc[0]
    print(f'Test: {row["title"][:70]}')
    print(json.dumps(assess_drought(row['title'], row['full_text']), indent=2))

## Step 7: Batch Assessment

In [ ]:
if HF_TOKEN == 'YOUR_HF_TOKEN_HERE':
    print('Set HF_TOKEN in Step 2 first.')
else:
    assessments = []
    for _, row in tqdm(df_climate.iterrows(), total=len(df_climate)):
        r = assess_drought(row['title'], row['full_text'])
        r.update({'title': row['title'], 'url': row['url'], 'source': row['source'], 'published': row['published']})
        assessments.append(r)
        time.sleep(1)
    df_assessed = pd.DataFrame(assessments)
    df_drought  = df_assessed[df_assessed['is_drought'] == True].reset_index(drop=True)
    print(f'Drought: {len(df_drought)} | Not drought: {(df_assessed["is_drought"]==False).sum()} | Errors: {df_assessed["is_drought"].isna().sum()}')

## Step 8: View & Export

In [ ]:
import os, matplotlib.pyplot as plt
os.makedirs('../data', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')

if 'df_drought' in dir() and len(df_drought) > 0:
    df_assessed.to_csv(f'../data/assessed_{ts}.csv', index=False)
    df_drought.to_csv(f'../data/drought_reports_{ts}.csv', index=False)
    df_drought.to_json(f'../data/drought_reports_{ts}.json', orient='records', indent=2)
    print(f'Saved {len(df_drought)} drought reports.')

    cols = ['title','location','severity','confidence','key_quote','source']
    show = [c for c in cols if c in df_drought.columns]
    pd.set_option('display.max_colwidth', 80)
    display(df_drought[show])

    if 'severity' in df_drought.columns:
        df_drought['severity'].value_counts().plot(kind='bar', color='tomato', title='Drought Severity')
        plt.tight_layout(); plt.show()
elif 'df_climate' in dir():
    df_climate.to_csv(f'../data/climate_filtered_{ts}.csv', index=False)
    print(f'No LLM assessment. Saved {len(df_climate)} climate-filtered articles to CSV.')
    df_climate[['title','source']].head(10)
else:
    print('Run Steps 3-4 first.')

## Next Steps

- Get HF token → enable LLM assessment (Steps 6-7)
- Add region queries: `Africa drought`, `India monsoon failure`
- Schedule daily run via cron
- Fine-tune ClimateBERT on drought-specific labels
- Build Streamlit dashboard